# Data Loading

In [59]:
with open('data/small_vocab_en', 'r') as f:
    eng_sentences = f.read().split('\n')
    
with open('data/small_vocab_fr', 'r') as f:
    fre_sentences = f.read().split('\n')

print('Dataset Loaded')

Dataset Loaded


In [60]:
print(eng_sentences[0:5],"...eng sentences...")
print(fre_sentences[0:5],"...french sentences..")
print(len(eng_sentences),"...length of english sentences...")
print(len(fre_sentences),"...length of french sentences....")

['new jersey is sometimes quiet during autumn , and it is snowy in april .', 'the united states is usually chilly during july , and it is usually freezing in november .', 'california is usually quiet during march , and it is usually hot in june .', 'the united states is sometimes mild during june , and it is cold in september .', 'your least liked fruit is the grape , but my least liked is the apple .'] ...eng sentences...
["new jersey est parfois calme pendant l' automne , et il est neigeux en avril .", 'les Ã©tats-unis est gÃ©nÃ©ralement froid en juillet , et il gÃ¨le habituellement en novembre .', 'california est gÃ©nÃ©ralement calme en mars , et il est gÃ©nÃ©ralement chaud en juin .', 'les Ã©tats-unis est parfois lÃ©gÃ¨re en juin , et il fait froid en septembre .', 'votre moins aimÃ© fruit est le raisin , mais mon moins aimÃ© est la pomme .'] ...french sentences..
137861 ...length of english sentences...
137861 ...length of french sentences....


# Pre-Processing  

In [61]:
import numpy as np
import io
import unicodedata
import re
from tqdm import tqdm

def unicode_to_ascii(s) :
  """
  Unicode to ascii conversion
  """
  return ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')

def cleanhtml(raw_html) :
  """
  Function to clean html tags and numbers
  """
  cleanr = re.compile('<.*?>|&([a-z0-9]+|#[0-9]{1,6}|#x[0-9a-f]{1,6});')
  cleantext = re.sub(cleanr, '', raw_html)
  return cleantext

def cleanString(incomingString):
    """
      Function to clean unwanted symbol from text
    """
    newstring = incomingString
    newstring = newstring.replace("!","")
    newstring = newstring.replace("@","")
    newstring = newstring.replace("#","")
    newstring = newstring.replace("$","")
    newstring = newstring.replace("%","")
    newstring = newstring.replace("^","")
    newstring = newstring.replace("&","and")
    newstring = newstring.replace("*","")
    newstring = newstring.replace("(","")
    newstring = newstring.replace(")","")
    newstring = newstring.replace("+","")
    newstring = newstring.replace("=","")
    newstring = newstring.replace("?","")
    newstring = newstring.replace("\'","")
    newstring = newstring.replace("\"","")
    newstring = newstring.replace("{","")
    newstring = newstring.replace("}","")
    newstring = newstring.replace("[","")
    newstring = newstring.replace("]","")
    newstring = newstring.replace("<","")
    newstring = newstring.replace(">","")
    newstring = newstring.replace("~","")
    newstring = newstring.replace("`","")
    newstring = newstring.replace(":","")
    newstring = newstring.replace(";","")
    newstring = newstring.replace("|","")
    newstring = newstring.replace("\\","")
    newstring = newstring.replace("/","")     
    return ' '.join(newstring.split())

def preprocess_string(data) :
  """
  This function calls other
  preprocessing function for
  cleaning data
  """
  data = unicode_to_ascii(data)
  #Remove html
  data = cleanhtml(data)
  #Remove unwanted symbols
  data = cleanString(data)
  return data


def start_preprocessing(data,lang):
    print("..started preprocessing..."+lang)
    preproc_data_list = []
    for val in tqdm(data):
        preproc_data=preprocess_string(val)
        preproc_data_list.append(preproc_data)
    return preproc_data_list    

In [62]:
eng_preproc_sentence = start_preprocessing(eng_sentences,"eng")
fre_preproc_senetence = start_preprocessing(fre_sentences,"french")

  2%|█▋                                                                       | 3157/137861 [00:00<00:04, 28443.38it/s]

..started preprocessing...eng


  2%|█▍                                                                       | 2780/137861 [00:00<00:05, 25848.98it/s]

..started preprocessing...french


100%|███████████████████████████████████████████████████████████████████████| 137861/137861 [00:05<00:00, 26892.43it/s]


# converting english to french

In [63]:
# Putting the start and end words in the french sentances

eng_preproc_sentence = [x.lower() for x in eng_preproc_sentence]
fre_preproc_senetence = [x.lower() for x in fre_preproc_senetence]
fre_preproc_senetence = ["start " + x + " end" for x in fre_preproc_senetence ]

In [64]:
eng_preproc_sentence = eng_preproc_sentence[0:1000] 
fre_preproc_senetence = fre_preproc_senetence[0:1000] 

In [65]:
from sklearn.model_selection import train_test_split
X=eng_preproc_sentence
Y=fre_preproc_senetence
X_train, X_test, y_train, y_test = train_test_split(X,Y,test_size = 0.1)
len(X_train),len(y_train), len(X_test), len(y_test)

(900, 900, 100, 100)

In [66]:
def Max_length(data):
  max_length_ = max([len(x.split(' ')) for x in data])
  return max_length_

#Training data
max_length_english = Max_length(X_train)
max_length_french = Max_length(y_train)

#Test data
max_length_english_test = Max_length(X_test)
max_length_french_test = Max_length(y_test)

print(max_length_english_test,"...max english length test..")
print(max_length_french_test,"....max french  length test..")

print(max_length_english,"...max english length train..")
print(max_length_french,"....max french  length train..")

17 ...max english length test..
21 ....max french  length test..
17 ...max english length train..
23 ....max french  length train..


# Tokenization of data

In [67]:
from sklearn.utils import shuffle
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import LSTM, Input, Dense,Embedding, Concatenate, TimeDistributed,Attention
from tensorflow.keras.models import Model,load_model, model_from_json
from tensorflow.keras.utils import plot_model
from tensorflow.keras.preprocessing.text import one_hot, Tokenizer
from tensorflow.keras.callbacks import EarlyStopping
import pickle as pkl
import numpy as np

englishTokenizer = Tokenizer()
englishTokenizer.fit_on_texts(X_train)
Eword2index = englishTokenizer.word_index
vocab_size_source = len(Eword2index) + 1

X_train = englishTokenizer.texts_to_sequences(X_train)
X_train = pad_sequences(X_train, maxlen=max_length_english, padding='post')

X_test = englishTokenizer.texts_to_sequences(X_test)
X_test = pad_sequences(X_test, maxlen = max_length_english, padding='post')


frenchTokenizer = Tokenizer()
frenchTokenizer.fit_on_texts(y_train)
Fword2index = frenchTokenizer.word_index
vocab_size_target = len(Fword2index) + 1

y_train = frenchTokenizer.texts_to_sequences(y_train)
y_train = pad_sequences(y_train, maxlen=max_length_french, padding='post')

y_test = frenchTokenizer.texts_to_sequences(y_test)
y_test = pad_sequences(y_test, maxlen = max_length_french, padding='post')


print(vocab_size_source,"...source vocab size...") 
print(vocab_size_target,"...target vocab size...")

166 ...source vocab size...
250 ...target vocab size...


In [68]:
X_train[0]

array([39, 11, 12, 13,  1,  5, 85,  6, 49, 11, 12,  1,  5, 86,  0,  0,  0])

# Model Training

In [107]:
from tensorflow.keras.layers import Attention,AdditiveAttention
from keras import backend as K 
K.clear_session() 
latent_dim = 512

In [108]:
# Encoder 
encoder_inputs = Input(shape=(max_length_english,)) 
enc_emb = Embedding(vocab_size_source, latent_dim,trainable=True)(encoder_inputs) 

#LSTM 1 
encoder_lstm1 = LSTM(latent_dim,return_sequences=True,return_state=True) 
encoder_output1, state_h1, state_c1 = encoder_lstm1(enc_emb) 

#LSTM 2 
encoder_lstm2 = LSTM(latent_dim,return_sequences=True,return_state=True) 
encoder_output2, state_h2, state_c2 = encoder_lstm2(encoder_output1) 

#LSTM 3 
encoder_lstm3=LSTM(latent_dim, return_state=True, return_sequences=True) 
encoder_outputs, state_h, state_c= encoder_lstm3(encoder_output2) 

# Set up the decoder. 
decoder_inputs = Input(shape=(None,)) 
dec_emb_layer = Embedding(vocab_size_target, latent_dim,trainable=True) 
dec_emb = dec_emb_layer(decoder_inputs) 

#LSTM using encoder_states as initial state
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True) 
decoder_outputs,_,_ = decoder_lstm(dec_emb,initial_state=[state_h, state_c]) 

# Concat decoder LSTM output with itself
decoder_concat_input = Concatenate(axis=-1, name='concat_layer')([decoder_outputs, decoder_outputs])

#Dense layer
decoder_dense = TimeDistributed(Dense(vocab_size_target, activation='softmax')) 
decoder_outputs = decoder_dense(decoder_concat_input) 

# Define the model
model = Model([encoder_inputs, decoder_inputs], decoder_outputs) 
model.summary()

Model: "model"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, 17)]         0                                            
__________________________________________________________________________________________________
embedding (Embedding)           (None, 17, 512)      84992       input_1[0][0]                    
__________________________________________________________________________________________________
lstm (LSTM)                     [(None, 17, 512), (N 2099200     embedding[0][0]                  
__________________________________________________________________________________________________
input_2 (InputLayer)            [(None, None)]       0                                            
______________________________________________________________________________________________

In [109]:
model.compile(optimizer='rmsprop', loss='sparse_categorical_crossentropy',
              metrics=['accuracy'], experimental_run_tf_function=False)
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1,patience=3)
#tf.config.experimental_run_functions_eagerly(True)
history = model.fit([X_train, y_train[:,:-1]], y_train.reshape(y_train.shape[0], y_train.shape[1],1)[:,1:], 
                    epochs=5, 
                    callbacks=[es],
                    batch_size=64,
                    validation_data = ([X_test, y_test[:,:-1]], y_test.reshape(y_test.shape[0], y_test.shape[1], 1)[:,1:]))

Epoch 1/5
15/15 [==============================] - 28s 1s/step - loss: 3.6754 - accuracy: 0.3699 - val_loss: 2.4581 - val_accuracy: 0.4682
Epoch 2/5
15/15 [==============================] - 18s 1s/step - loss: 2.3707 - accuracy: 0.4942 - val_loss: 2.1861 - val_accuracy: 0.4968
Epoch 3/5
15/15 [==============================] - 19s 1s/step - loss: 2.0056 - accuracy: 0.5527 - val_loss: 1.8383 - val_accuracy: 0.5600
Epoch 4/5
15/15 [==============================] - 18s 1s/step - loss: 1.5860 - accuracy: 0.6156 - val_loss: 1.6714 - val_accuracy: 0.5973
Epoch 5/5
15/15 [==============================] - 18s 1s/step - loss: 1.3239 - accuracy: 0.6512 - val_loss: 1.4153 - val_accuracy: 0.6250


In [110]:
y_train[:,:-1][0]

array([ 2, 48, 19, 18,  1, 14,  9, 86,  6, 28, 14, 18,  1,  9, 87,  3,  0,
        0,  0,  0,  0,  0])

In [111]:
y_train[0]

array([ 2, 48, 19, 18,  1, 14,  9, 86,  6, 28, 14, 18,  1,  9, 87,  3,  0,
        0,  0,  0,  0,  0,  0])

In [112]:
 y_train.reshape(y_train.shape[0], y_train.shape[1],1)[:,1:][0]

array([[48],
       [19],
       [18],
       [ 1],
       [14],
       [ 9],
       [86],
       [ 6],
       [28],
       [14],
       [18],
       [ 1],
       [ 9],
       [87],
       [ 3],
       [ 0],
       [ 0],
       [ 0],
       [ 0],
       [ 0],
       [ 0],
       [ 0]])

In [113]:
model_json = model.to_json()
with open("FET_model.json", "w") as json_file:
    json_file.write(model_json)
# serialize weights to HDF5
model.save_weights("FET_model_weight.h5")
print("Saved model to disk")

# loading the model architecture and asigning the weights
json_file = open('FET_model.json', 'r')
loaded_model_json = json_file.read()
json_file.close()
model_loaded = model_from_json(loaded_model_json 
                               #, #custom_objects={'AttentionLayer': AttentionLayer}
                              )
# load weights into new model
model_loaded.load_weights("FET_model_weight.h5")

Saved model to disk


In [114]:
latent_dim=512
# encoder inference
encoder_inputs = model_loaded.input[0]  #loading encoder_inputs
encoder_outputs, state_h, state_c = model_loaded.layers[6].output #loading encoder_outputs

print(encoder_outputs.shape)

encoder_model = Model(inputs=encoder_inputs,outputs=[encoder_outputs, state_h, state_c])

# decoder inference
# Below tensors will hold the states of the previous time step
decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_hidden_state_input = Input(shape=(17,latent_dim))

# Get the embeddings of the decoder sequence
decoder_inputs = model_loaded.layers[3].output

print(decoder_inputs.shape)
dec_emb_layer = model_loaded.layers[5]

dec_emb2= dec_emb_layer(decoder_inputs)

# To predict the next word in the sequence, set the initial states to the states from the previous time step
decoder_lstm = model_loaded.layers[7]
decoder_outputs2, state_h2, state_c2 = decoder_lstm(dec_emb2, initial_state=[decoder_state_input_h, decoder_state_input_c])

# Concat decoder LSTM output with itself
concate = model_loaded.layers[8]
decoder_inf_concat = concate([decoder_outputs2,decoder_outputs2])
#decoder_inf_concat = Concatenate(axis=1)([decoder_hidden_state_input, decoder_outputs2])

# A dense softmax layer to generate prob dist. over the target vocabulary
decoder_dense = model_loaded.layers[9]
decoder_outputs2 = decoder_dense(decoder_inf_concat)


# Final decoder model
decoder_model = Model(
[decoder_inputs] + [decoder_hidden_state_input,decoder_state_input_h, decoder_state_input_c],
[decoder_outputs2] + [state_h2, state_c2])


(None, 17, 512)
(None, None)


In [115]:
Eindex2word = englishTokenizer.index_word
Findex2word = frenchTokenizer.index_word

In [116]:
def decode_sequence(input_seq):
    # Encode the input as state vectors.
    #print(input_seq,".....input_seq.....")
    e_out, e_h, e_c = encoder_model.predict(input_seq)
    
    # Generate empty target sequence of length 1.
    target_seq = np.zeros((1,1))
    #print(target_seq,"...target_seq....")
    
    # Chose the 'start' word as the first word of the target sequence
    target_seq[0, 0] = Fword2index['start']
    #print(target_seq,"...target_seq after start....")
    
    stop_condition = False
    decoded_sentence = ''
    while not stop_condition:
        output_tokens, h, c = decoder_model.predict([target_seq] + [e_out, e_h, e_c])

        # Sample a token
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        if sampled_token_index == 0:
          break
        else:
          sampled_token = Findex2word[sampled_token_index]

          if(sampled_token!='end'):
              decoded_sentence += ' '+sampled_token

              # Exit condition: either hit max length or find stop word.
              if (sampled_token == 'end' or len(decoded_sentence.split()) >= (26-1)):
                  stop_condition = True

          # Update the target sequence (of length 1).
          target_seq = np.zeros((1,1))
          target_seq[0, 0] = sampled_token_index

          # Update internal states
          e_h, e_c = h, c

    return decoded_sentence

In [117]:
def seq2summary(input_seq):
    newString=''
    for i in input_seq:
      if((i!=0 and i!=Fword2index['start']) and i!=Fword2index['end']):
        newString=newString+Findex2word[i]+' '
    return newString

def seq2text(input_seq):
    newString=''
    for i in input_seq:
      if(i!=0):
        newString=newString+Eindex2word[i]+' '
    return newString

In [118]:
for i in range(10):
  
  print("Review:",seq2text(X_test[i]))
  print("Original summary:",seq2summary(y_test[i]))
  print("Predicted summary:",decode_sequence(X_test[i].reshape(1,17)))
  print("\n")

Review: he dislikes grapes grapefruit and bananas 
Original summary: il da©teste les raisins le pamplemousse et les bananes 
Predicted summary:  elle aime pas pas citrons citrons citrons et les citrons citrons


Review: california is busy during june but it is usually hot in january 
Original summary: californie est occupa© en juin mais il est ga©na©ralement chaud en janvier 
Predicted summary:  new jersey est ga©na©ralement chaud en juin mais il est ga©na©ralement chaud en juin


Review: our most loved fruit is the strawberry but his most loved is the mango 
Original summary: nos fruits le plus aima© est la fraise mais son plus aima© est la mangue 
Predicted summary:  new jersey est ga©na©ralement chaud en juin mais il est ga©na©ralement chaud en juin


Review: paris is usually dry during april and it is usually nice in summer 
Original summary: paris est ga©na©ralement sec en avril et il est ga©na©ralement agra©able en a©ta© 
Predicted summary:  new jersey est ga©na©ralement chaud en